# 01. 래스터 데이터 다루기 — rasterio

위성영상·고도모델 같은 **래스터(raster)** 데이터는 격자(픽셀)로 이루어지며, 각 픽셀이 지리 위치와 연결됩니다.
`rasterio`(GDAL 기반)로 래스터를 읽고, 좌표계(CRS)·변환정보를 다루며 시각화합니다.

1. 래스터의 구성 (밴드·CRS·transform)
2. 합성 위성영상 만들기·저장 (GeoTIFF)
3. 읽기·메타데이터 확인·시각화

> 이 노트북은 개념 이해를 위해 합성 래스터를 생성합니다. 실습에서는 `rasterio.open('영상.tif')`로 실제 위성영상을 불러옵니다.

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import matplotlib.pyplot as plt

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
print('rasterio', rasterio.__version__)

## 1. 래스터 구성 요소

래스터는 다음으로 정의됩니다.
- **밴드(band)**: 위성영상은 보통 여러 밴드(빨강·초록·파랑·근적외선 등)를 가집니다.
- **CRS(좌표계)**: 픽셀 좌표를 지구상 위치로 연결 (예: EPSG:32618 = UTM Zone 18N).
- **transform**: 픽셀 인덱스를 지리 좌표로 변환하는 정보 (원점·픽셀 크기).

In [ ]:
# 합성 3밴드 위성영상 (높이, 너비)
H, W = 200, 200
yy, xx = np.mgrid[0:H, 0:W]
band_red = (100 + 50*np.sin(xx/20)).astype('float32')
band_green = (120 + 40*np.cos(yy/25)).astype('float32')
band_nir = (180 + 60*np.sin((xx+yy)/30)).astype('float32')
data = np.stack([band_red, band_green, band_nir])   # (3, H, W)

# 지리 정보: 원점(좌상단) UTM 좌표, 픽셀 30m
transform = from_origin(500000, 4100000, 30, 30)
crs = CRS.from_epsg(32618)   # UTM Zone 18N
print('데이터 shape (밴드,높이,너비):', data.shape)
print('CRS:', crs)

## 2. GeoTIFF로 저장

지리 정보(CRS·transform)와 함께 GeoTIFF로 저장합니다. GeoTIFF는 위성영상의 표준 포맷입니다.

In [ ]:
path = '/workspace/sample_scene.tif'
with rasterio.open(path, 'w', driver='GTiff',
                   height=H, width=W, count=3,
                   dtype='float32', crs=crs, transform=transform) as dst:
    dst.write(data)
print('저장 완료:', path)

## 3. 읽기·메타데이터·시각화

저장한 래스터를 읽어 메타데이터를 확인하고, 밴드를 시각화합니다.

In [ ]:
with rasterio.open(path) as src:
    print('밴드 수:', src.count)
    print('크기 (너비 x 높이):', src.width, 'x', src.height)
    print('CRS:', src.crs)
    print('경계(bounds):', src.bounds)
    red = src.read(1); green = src.read(2); nir = src.read(3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, band, name in zip(axes, [red, green, nir], ['Red', 'Green', 'NIR']):
    im = ax.imshow(band, cmap='gray'); ax.set_title(f'{name} 밴드'); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

### 정리

- 래스터의 핵심 구성(밴드·CRS·transform)을 이해하고, rasterio로 GeoTIFF를 저장·로드했습니다.
- 위성영상이 단순 이미지와 다른 점은 **지리 좌표 정보**가 함께 있다는 것입니다.
- 다음 노트북에서는 폴리곤 같은 **벡터 데이터**를 다룹니다.